# preborn_tests — invariant gates + size report for the synthetic PREBORN copy

Run AFTER `preborn_pipeline role=derive`. Validates the `synth_*` finals in
`8_dev.synth_preborn` (or the configured FINAL schema) against PREBORN's
structural invariants — the same ones the real pipeline asserts — plus a
per-table size report vs the real run schema. Raises on any hard-gate failure.

In [0]:
%run ./preborn_config

In [0]:
FINAL = f"{FINAL_CATALOG}.{FINAL_SCHEMA}"
PFX = SYNTH_PREFIX

_hard, _soft = [], []
def gate(name, passed, detail="", soft=False):
    (_soft if soft else _hard).append((name, bool(passed), str(detail)))
    tag = "SOFT" if soft else ("PASS" if passed else "FAIL")
    print(f"[{tag}] {name}" + (f" — {detail}" if detail else ""))

def q1(sql):
    return spark.sql(sql).first()[0]

def T(t):
    return f"{FINAL}.{PFX}{t}"

In [0]:
# ---- existence / row counts ----
CLINICAL = ["person", "observation_period", "pregnancy", "infant", "visit_occurrence",
            "visit_detail", "measurement", "observation", "condition_occurrence",
            "procedure_occurrence", "episode", "episode_event", "fact_relationship",
            "cdm_source"]
VOCAB = ["concept", "concept_relationship", "concept_ancestor", "concept_synonym",
         "vocabulary", "domain", "concept_class", "relationship"]
counts = {}
for t in CLINICAL + VOCAB:
    try:
        counts[t] = spark.table(T(t)).count()
    except Exception as e:
        counts[t] = None
        gate(f"exists {t}", False, str(e)[:120])
print("counts:", {k: v for k, v in counts.items()})

# ---- role counts ----
n_mother = q1(f"SELECT COUNT(*) FROM {T('person')} WHERE person_role='mother'")
n_infant = q1(f"SELECT COUNT(*) FROM {T('person')} WHERE person_role='infant'")
gate("person has both roles", n_mother > 0 and n_infant > 0, f"mother={n_mother} infant={n_infant}")

# ---- infant_id == person_id (bijection with infant persons) ----
gate("infant_id == person_id (all infants are persons)",
     q1(f"""SELECT COUNT(*) FROM {T('infant')} i
            WHERE NOT EXISTS (SELECT 1 FROM {T('person')} p
              WHERE p.person_id=i.infant_id AND p.person_role='infant')""") == 0,
     "every infant.infant_id resolves to an infant person")
gate("infant persons <-> infant rows bijection",
     q1(f"""SELECT COUNT(*) FROM {T('person')} p WHERE p.person_role='infant'
            AND NOT EXISTS (SELECT 1 FROM {T('infant')} i WHERE i.infant_id=p.person_id)""") == 0,
     "every infant person has an infant extension row")

# ---- pk uniqueness ----
for t, pk in [("person", "person_id"), ("pregnancy", "pregnancy_id"),
              ("infant", "infant_id"), ("visit_occurrence", "visit_occurrence_id"),
              ("measurement", "measurement_id"), ("observation", "observation_id"),
              ("condition_occurrence", "condition_occurrence_id"),
              ("procedure_occurrence", "procedure_occurrence_id"),
              ("episode", "episode_id"), ("observation_period", "observation_period_id"),
              ("visit_detail", "visit_detail_id")]:
    tot = q1(f"SELECT COUNT(*) FROM {T(t)}")
    dist = q1(f"SELECT COUNT(DISTINCT `{pk}`) FROM {T(t)}")
    gate(f"pk_unique {t}.{pk}", tot == dist, f"{tot} rows / {dist} distinct")

# ---- orphan FKs ----
def orphan(child, fk, parent, ppk):
    return q1(f"""SELECT COUNT(*) FROM {T(child)} c WHERE c.`{fk}` IS NOT NULL
      AND NOT EXISTS (SELECT 1 FROM {T(parent)} p WHERE p.`{ppk}`=c.`{fk}`)""")
gate("fk pregnancy.person_id -> person", orphan("pregnancy", "person_id", "person", "person_id") == 0)
gate("fk infant.pregnancy_id -> pregnancy", orphan("infant", "pregnancy_id", "pregnancy", "pregnancy_id") == 0)
for t in ["visit_occurrence", "measurement", "observation", "condition_occurrence",
          "procedure_occurrence", "episode", "observation_period", "visit_detail"]:
    gate(f"fk {t}.person_id -> person", orphan(t, "person_id", "person", "person_id") == 0)
for t in ["measurement", "observation", "condition_occurrence", "procedure_occurrence"]:
    gate(f"fk {t}.visit_occurrence_id -> visit", orphan(t, "visit_occurrence_id", "visit_occurrence", "visit_occurrence_id") == 0)
gate("fk visit_detail.visit_occurrence_id -> visit",
     orphan("visit_detail", "visit_occurrence_id", "visit_occurrence", "visit_occurrence_id") == 0)
for t in ["visit_occurrence", "measurement", "observation", "condition_occurrence",
          "procedure_occurrence", "episode"]:
    gate(f"fk {t}.pregnancy_id -> pregnancy (or null)",
         orphan(t, "pregnancy_id", "pregnancy", "pregnancy_id") == 0)

# ---- episode: one per pregnancy ----
gate("episode 1:1 with pregnancy", counts["episode"] == counts["pregnancy"],
     f"episode={counts['episode']} pregnancy={counts['pregnancy']}")

# ---- episode_event: 5 source tables only + event_id resolves ----
srcs = [r[0] for r in spark.sql(f"SELECT DISTINCT source_table FROM {T('episode_event')}").collect()]
gate("episode_event exactly 5 source tables", set(srcs) == set(EPISODE_EVENT_FIELDS.keys()),
     f"{sorted(srcs)}")
gate("episode_event.episode_id resolves",
     orphan("episode_event", "episode_id", "episode", "episode_id") == 0)
for st, (fc, idcol) in EPISODE_EVENT_FIELDS.items():
    miss = q1(f"""SELECT COUNT(*) FROM {T('episode_event')} ee
      WHERE ee.source_table='{st}'
        AND NOT EXISTS (SELECT 1 FROM {T(st)} f WHERE f.`{idcol}`=ee.event_id)""")
    gate(f"episode_event.event_id resolves in {st}", miss == 0, f"{miss} unresolved")

# ---- observation_period: one per person ----
gate("observation_period 1:1 with person", counts["observation_period"] == counts["person"],
     f"op={counts['observation_period']} person={counts['person']}")

# ---- fact_relationship: endpoints resolve + semantics ----
for fk in ["fact_id_1", "fact_id_2"]:
    gate(f"fact_relationship.{fk} -> person", orphan("fact_relationship", fk, "person", "person_id") == 0)
fr_by = {r[0]: r[1] for r in spark.sql(
    f"SELECT relationship_concept_id, COUNT(*) c FROM {T('fact_relationship')} GROUP BY 1").collect()}
gate("fact_relationship mother/child symmetric",
     fr_by.get(REL_MOTHER, 0) == fr_by.get(REL_CHILD, -1),
     f"mother={fr_by.get(REL_MOTHER)} child={fr_by.get(REL_CHILD)}")
# twins share a pregnancy
bad_twin = q1(f"""SELECT COUNT(*) FROM {T('fact_relationship')} fr
  JOIN {T('infant')} a ON a.infant_id=fr.fact_id_1
  JOIN {T('infant')} b ON b.infant_id=fr.fact_id_2
  WHERE fr.relationship_concept_id={REL_TWIN} AND a.pregnancy_id<>b.pregnancy_id""")
gate("twins share a pregnancy", bad_twin == 0, f"{bad_twin} twin pairs across pregnancies")

# ---- PII 100% null ----
for c in _PII.get("person", []):
    nn = q1(f"SELECT COUNT(*) FROM {T('person')} WHERE `{c}` IS NOT NULL")
    gate(f"pii_null person.{c}", nn == 0, f"{nn} non-null")

# ---- site_id constant ----
for t in ["person", "pregnancy", "visit_occurrence", "measurement", "episode"]:
    d = q1(f"SELECT COUNT(DISTINCT site_id) FROM {T(t)}")
    only = q1(f"SELECT MIN(site_id) FROM {T(t)}")
    gate(f"site_id constant {t}", d == 1 and only == SITE_ID, f"distinct={d} val={only}")

# ---- vocab FK closure (spot) ----
dang = q1(f"""SELECT COUNT(*) FROM {T('concept_relationship')} r
  LEFT JOIN {T('concept')} c ON c.concept_id=r.concept_id_1 WHERE c.concept_id IS NULL""")
gate("vocab concept_relationship.c1 in concept", dang == 0, f"{dang} dangling")
# every concept_id used in facts is in the bundled concept table
used_missing = q1(f"""
  WITH used AS (
    SELECT DISTINCT measurement_concept_id AS cid FROM {T('measurement')}
    UNION SELECT observation_concept_id FROM {T('observation')}
    UNION SELECT condition_concept_id FROM {T('condition_occurrence')}
    UNION SELECT visit_concept_id FROM {T('visit_occurrence')})
  SELECT COUNT(*) FROM used u WHERE u.cid IS NOT NULL AND u.cid>0
    AND NOT EXISTS (SELECT 1 FROM {T('concept')} c WHERE c.concept_id=u.cid)""")
gate("all used fact concept_ids in bundled concept", used_missing == 0, f"{used_missing} missing")

In [0]:
# ---- SIZE REPORT vs real run schema (tolerance-based, informational) ----
print("\n=== size report: synth vs real ===")
size_rows = []
for t in ["person", "observation_period", "pregnancy", "infant", "visit_occurrence",
          "visit_detail", "measurement", "observation", "condition_occurrence",
          "procedure_occurrence", "episode", "episode_event", "fact_relationship"]:
    real = TARGET.get(t)
    syn = counts.get(t)
    ratio = (syn / real) if (real and syn is not None) else None
    size_rows.append((t, syn, real, round(ratio, 3) if ratio else None))
    print(f"  {t:22s} synth={syn} real={real} ratio={round(ratio,3) if ratio else None}")

In [0]:
n_fail = sum(1 for _, p, _ in _hard if not p)
n_warn = sum(1 for _, p, _ in _soft if not p)
print(f"\n{'PASS' if n_fail==0 else 'FAIL'}: {len(_hard)} hard gates, {n_fail} failed; {n_warn} soft warnings")
if n_fail:
    fails = [n for n, p, d in _hard if not p]
    raise AssertionError(f"{n_fail} hard gate(s) failed: {fails}")
print("ALL HARD GATES PASSED")